# BERTurk Fine-Tuning for Call Center Quality Score Prediction

This notebook fine-tunes a BERTurk-based regression model to predict a call center quality score between 0 and 100 from Turkish call center conversations.

Before this stage, the conversation texts were prepared, synthetic medium/low-quality examples were generated, and each call was assigned a final_quality_score using an LLM-based pseudo-labeling approach.

In this notebook, the prepared dataset is used to fine-tune a Turkish BERT model for call quality score prediction.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
BASE_DIR = "/content/drive/MyDrive/bitirme_dataset"
DATA_PATH = f"{BASE_DIR}/final_data.csv"
SAVE_PATH = f"{BASE_DIR}/berturk_call_quality_regression"

In [3]:
import torch

In [4]:
print('CUDA available:', torch.cuda.is_available())
print('GPU:',torch.cuda.get_device_name(0)) if torch.cuda.is_available() else print('No GPU')

CUDA available: True
GPU: Tesla T4


In [5]:
import pandas as pd
data = pd.read_csv(DATA_PATH)

data = data.dropna(subset=["whole_text", "final_quality_score"]).copy()

data["whole_text"] = data["whole_text"].astype(str)
data["final_quality_score"] = data["final_quality_score"].astype(float)

data = data[
    (data["final_quality_score"] >= 0) &
    (data["final_quality_score"] <= 100)
].copy()

print(data.shape)
data.head()

(2695, 8)


,id,original_id,category,source_type,degradation_level,expected_quality_class,whole_text,final_quality_score
0,2,2,Finansal Hizmetler,original,none,Good,"representative: Merhaba, size nasıl yardımcı o...",56.50
1,3,3,Finansal Hizmetler,original,none,Good,"customer: Merhaba, BP monitörü satın almaya ça...",82.00
2,4,4,Finansal Hizmetler,original,none,Good,"customer: Merhaba, son siparişimde bir fatura ...",92.25
3,5,5,Finansal Hizmetler,original,none,Good,"representative: Merhaba, benim adım John. Size...",86.75
4,6,6,Finansal Hizmetler,original,none,Good,"customer: Merhaba, BrownBox'tan aldığım telefo...",90.25


Once your Google Drive is mounted, you can navigate to your files. For example, if you uploaded a CSV file named `my_data.csv` to a folder named `Colab Notebooks` in your Drive, you could access it like this:

```python
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/my_data.csv')
display(df.head())
```

Please let me know the path to your data file in Google Drive, and I can help you load it.

In [6]:
data.drop('expected_quality_class',axis=1,inplace=True)


## Creating Quality Classes for Stratified Split

The main modeling task is regression. The model predicts the continuous `final_quality_score` value between 0 and 100.

However, temporary quality classes are created to perform a more balanced train-test split. These classes are not used as the main prediction target.

The score ranges are defined as follows:

- `excellent`: 85–100
- `good`: 70–84
- `average`: 55–69
- `poor`: 0–54

These classes are used only for stratification during train-test splitting.


In [7]:
def score_to_class(score):
    if score >= 85:
        return "excellent"
    elif score >= 70:
        return "good"
    elif score >= 55:
        return "average"
    else:
        return "poor"
data['quality_class'] = data['final_quality_score'].apply(score_to_class)


In [8]:
print(data["quality_class"].value_counts())

quality_class
poor         1116
good          602
average       561
excellent     416
Name: count, dtype: int64


## Train-Test Split

The dataset is split into training and test sets.

The input variable is:

- `whole_text`

The target variable is:

- `final_quality_score`

Although the task is regression, the split is stratified using the temporary `quality_class` column. This helps preserve a similar distribution of low, medium, and high-quality calls in both training and test sets.

In [43]:
from sklearn.model_selection import train_test_split

X = data[["whole_text","original_id"]]
y = data["final_quality_score"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=17,
    shuffle=True,
    stratify=data["quality_class"]
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (2156, 2)
Test: (539, 2)


In [44]:
aug_df= pd.read_csv(f"{BASE_DIR}/augmented_training_data.csv")
aug_df = aug_df[aug_df['base_original_id'].isin(X_train['original_id'])]
X_train= pd.concat([X_train['whole_text'],aug_df['whole_text']],axis=0,ignore_index=True)
y_train= pd.concat([y_train,aug_df['final_quality_score']],axis=0,ignore_index=True)
X_test=X_test['whole_text']
print(X_train.shape,y_train.shape)

(2666,) (2666,)


## Converting Pandas DataFrames to Hugging Face Dataset Format

Hugging Face Trainer works more conveniently with the Dataset format. Therefore, the training and test data are converted from pandas DataFrames to Hugging Face Dataset objects.

Each example contains:

- text: The call transcript
- label: The normalized quality score

The original quality score is divided by 100 and converted from the 0–100 range to the 0–1 range. This makes regression training more stable. During evaluation, predictions are converted back to the 0–100 scale.

In [45]:
from datasets import Dataset

train_df = pd.DataFrame({
    "text": X_train.values,
    "label": (y_train.values.astype(float) / 100.0)
})

test_df = pd.DataFrame({
    "text": X_test.values,
    "label": (y_test.values.astype(float) / 100.0)
})

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)

train_dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 2666
})

In [46]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "dbmdz/bert-base-turkish-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=1,
    problem_type="regression"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/251k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Loading BERTurk Tokenizer and Regression Model

The pre-trained Turkish BERT model dbmdz/bert-base-turkish-cased is loaded from Hugging Face.

The tokenizer converts raw text into token IDs that the model can process.

The model is loaded using AutoModelForSequenceClassification, but it is configured for regression by setting:

- num_labels=1
- problem_type="regression"

This means the model outputs a single numerical value instead of class probabilities.

In [47]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/2666 [00:00<?, ? examples/s]

Map:   0%|          | 0/539 [00:00<?, ? examples/s]

## Evaluation Metrics

The model is evaluated using regression metrics:

- MAE: (Mean Absolute Error): Average absolute prediction error
- RMSE: (Root Mean Squared Error): Penalizes larger errors more strongly
- R² : Measures how much of the score variance is explained by the model

Since the model is trained on normalized labels between 0 and 1, predictions and labels are multiplied by 100 before metric calculation. This allows the results to be interpreted on the original 0–100 quality score scale.

In [48]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    predictions = predictions.reshape(-1)
    labels = labels.reshape(-1)

    predictions_100 = np.clip(predictions * 100, 0, 100)
    labels_100 = labels * 100

    mae = mean_absolute_error(labels_100, predictions_100)
    rmse = np.sqrt(mean_squared_error(labels_100, predictions_100))
    r2 = r2_score(labels_100, predictions_100)

    return {
        "mae": mae,
        "rmse": rmse,
        "r2": r2
    }

## Training Arguments

The training configuration is defined using TrainingArguments.

Important parameters:

- learning_rate: Controls how fast the model updates its weights.
- per_device_train_batch_size: Number of training examples processed per batch on each device.
- num_train_epochs: Number of times the model sees the full training dataset.
- eval_strategy="epoch": Evaluates the model at the end of each epoch.
- save_strategy="epoch": Saves a checkpoint at the end of each epoch.
- load_best_model_at_end=True: Loads the checkpoint with the best validation performance at the end of training.
- metric_for_best_model="mae": Selects the best model based on the lowest MAE.
- greater_is_better=False: Indicates that lower MAE is better.

In [49]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=SAVE_PATH,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=8,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="mae",
    greater_is_better=False,
    report_to="none"
)

In [50]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [52]:
trainer.train()

Epoch,Training Loss,Validation Loss,Mae,Rmse,R2
1,0.023139,0.016769,9.760760,12.945996,0.800698
2,0.024617,0.013674,8.688499,11.691851,0.837442
3,0.020197,0.015445,9.538568,12.382024,0.817684
4,0.022625,0.016372,9.834757,12.712602,0.807819
5,0.020545,0.017018,10.097657,13.010333,0.798712


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Epoch,Training Loss,Validation Loss,Mae,Rmse,R2
1,0.023139,0.016769,9.760760,12.945996,0.800698
2,0.024617,0.013674,8.688499,11.691851,0.837442
3,0.020197,0.015445,9.538568,12.382024,0.817684
4,0.022625,0.016372,9.834757,12.712602,0.807819
5,0.020545,0.017018,10.097657,13.010333,0.798712
6,0.017141,0.009920,7.197282,9.940892,0.882485
7,0.016025,0.015155,9.441245,12.237875,0.821904
8,0.015057,0.012895,8.554084,11.308248,0.847934


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2672, training_loss=0.019679624040772815, metrics={'train_runtime': 2292.1558, 'train_samples_per_second': 9.305, 'train_steps_per_second': 1.166, 'total_flos': 5678938029539328.0, 'train_loss': 0.019679624040772815, 'epoch': 8.0})

In [53]:
bert_results = trainer.evaluate()
bert_results

Training Loss,Validation Loss,Epoch,Mae,Rmse,R2
0.015057,0.009920,8,7.197282,9.940892,0.882485


{'eval_loss': 0.009919976815581322,
 'eval_mae': 7.197281837463379,
 'eval_rmse': 9.940892054482193,
 'eval_r2': 0.8824854493141174}

## Final Evaluation

After training is completed, the best model is evaluated again on the test set.

The final results are compared with previous baseline models such as:

- CountVectorizer + Ridge Regression
- TF-IDF + Ridge Regression
- Sentence-BERT + Ridge Regression

The main comparison metrics are MAE, RMSE, and R².


In [54]:
import pandas as pd

COMPARISON_PATH = "/content/drive/MyDrive/bitirme_dataset/model_comparison_results.csv"

comparison_df = pd.read_csv(COMPARISON_PATH)

bert_result_row = {
    "model": "BERTurk Fine-Tuned Regression",
    "mae": bert_results["eval_mae"],
    "rmse": bert_results["eval_rmse"],
    "r2": bert_results["eval_r2"]
}

comparison_df = pd.concat(
    [comparison_df, pd.DataFrame([bert_result_row])],
    ignore_index=True
)

comparison_df.to_csv(COMPARISON_PATH)

In [56]:
SAVE_PATH = f"{BASE_DIR}/berturk_call_quality_regression/model"
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Model saved to:", SAVE_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/drive/MyDrive/bitirme_dataset/berturk_call_quality_regression/model


In [57]:
from transformers import AutoTokenizer

SAVE_PATH = "/content/drive/MyDrive/bitirme_dataset/berturk_call_quality_regression_augmented"

tokenizer = AutoTokenizer.from_pretrained(
    "dbmdz/bert-base-turkish-cased",
    use_fast=False
)

tokenizer.save_pretrained(SAVE_PATH)

print("Tokenizer saved again.")

Tokenizer saved again.
